In [ ]:
# --- Tile IDs (from your list) ---------------------------------------------
options = {
    'grassUp': 1,
    'grassTop': 2,
    'grassDown': 3,
    'grassLeftEdge': 4,
    'grassTopLeft': 5,
    'dirtFull': 6,
    'grassTopRight': 7,
    'grassRightEdge': 8,
    'dirtLeftWall': 9,
    'grassToStoneTop': 10,
    'stoneTop': 11,
    'stoneToGrassTop': 12,
    'dirtRightWall': 13,
}

# --- Edges: names of allowed neighbor blocks per side ----------------------
# Fill these with the neighbor NAMES you want. Use empty sets initially.
EDGES = {
    "grassUp":        dict(up=set(), right=set(), down=set(), left=set()),
    "grassTop":       dict(up=set(), right=set(), down=set(), left=set()),
    "grassDown":      dict(up=set(), right=set(), down=set(), left=set()),
    "grassLeftEdge":  dict(up=set(), right=set(), down=set(), left=set()),
    "grassLeftTop":   dict(up=set(), right=set(), down=set(), left=set()),
    "dirtFull":       dict(up=set(), right=set(), down=set(), left=set()),
    "grassRightTop":  dict(up=set(), right=set(), down=set(), left=set()),
    "grassRightEdge": dict(up=set(), right=set(), down=set(), left=set()),
    "dirtLeftWall":   dict(up=set(), right=set(), down=set(), left=set()),
    "grassToStoneTop":dict(up=set(), right=set(), down=set(), left=set()),
    "stoneTop":       dict(up=set(), right=set(), down=set(), left=set()),
    "stoneToGrassTop":dict(up=set(), right=set(), down=set(), left=set()),
    "dirtRightWall":  dict(up=set(), right=set(), down=set(), left=set()),
}

# Directions helpers for symmetry
OPPOSITE = {"up":"down", "right":"left", "down":"up", "left":"right"}

def connect(a: str, direction: str, *bs: str) -> None:
    """
    Add bidirectional adjacency: allow each b in `bs` to sit at `direction`
    from `a`, and `a` at the opposite side of each `b`.
    """
    for b in bs:
        EDGES[a][direction].add(b)
        EDGES[b][OPPOSITE[direction]].add(a)

def can_neighbor(a: str, b: str, direction: str) -> bool:
    """Is block `b` allowed at `direction` of block `a`?"""
    return b in EDGES[a][direction]

def allowed(a: str, direction: str):
    """Get all allowed neighbor names for tile `a` in `direction`."""
    return EDGES[a][direction]

# --- Example: populate a few relationships (delete/replace as you like) ----
connect("grassTop", "down", "dirtFull", "dirtLeftWall", "dirtRightWall")
connect("grassTop", "right", "grassTop", "grassToStoneTop", "grassLeftTop")
connect("grassTop", "left", "grassTop", "grassToStoneTop", "grassRightTop")

connect("grassUp", "right", "grassTop", "grassToStoneTop", "grassLeftTop")
connect("grassUp", "down", "grassLeftTop")

connect("grassDown", "left", "grassTop", "grassToStoneTop", "grassRightTop")
connect("grassDown", "down", "grassRightTop")


connect('stoneTop')




In [17]:
allowed('grassTop', 'down')


{'dirtFull', 'dirtLeftWall', 'dirtRightWall'}

In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Create hexagonal images with configurable diagonals and colors.

Usage examples (run this file):

- Example 1: single hex, only long diagonals, random colors.
- Example 2: 3x3 grid, all diagonals, random colors.
- Example 3: single hex, custom diagonal pairs + custom colors.

Edit the `if __name__ == "__main__":` section for quick experiments.
"""

import math
import random
from collections.abc import Iterable

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

Point = tuple[float, float]
Pair = tuple[int, int]  # vertex index pair (0..5)

# ----------------------------- Geometry helpers -----------------------------

def regular_hex_vertices(center: Point, r: float, flat_top: bool = True) -> list[Point]:
    """
    Return 6 vertices (x,y) of a regular hexagon in CCW order.

    flat_top=True -> vertices at angles 0°, 60°, ..., (flat sides top/bottom).
    flat_top=False -> pointy-top orientation (30°, 90°, ...).
    """
    cx, cy = center
    base_angle_deg = 0.0 if flat_top else 30.0
    verts = []
    for k in range(6):
        a = math.radians(base_angle_deg + 60.0 * k)
        verts.append((cx + r * math.cos(a), cy + r * math.sin(a)))
    return verts

def hex_center(verts: list[Point]) -> Point:
    """Centroid (for regular hex, equals the true center)."""
    x = sum(p[0] for p in verts) / 6.0
    y = sum(p[1] for p in verts) / 6.0
    return (x, y)

def diagonal_pairs(include: str = "all", pairs: Iterable[Pair] | None = None) -> list[Pair]:
    """
    Return pairs of vertex indices according to `include`:
      - "short" : step 2 (i -> i+2 mod 6), 6 diagonals
      - "long"  : step 3 (opposites, i -> i+3), 3 diagonals
      - "all"   : both (9 diagonals)
      - "custom": use `pairs` exactly (indices 0..5)
    """
    def canon(i, j):  # keep as (min,max) to avoid duplicates
        return (min(i % 6, j % 6), max(i % 6, j % 6))

    shorts = {canon(i, i + 2) for i in range(6)}
    longs  = {canon(i, i + 3) for i in range(6)}

    if include == "short":
        return sorted(shorts)
    if include == "long":
        return sorted(longs)
    if include == "all":
        return sorted(shorts | longs)
    if include == "custom":
        if not pairs:
            return []
        return sorted({canon(i, j) for i, j in pairs})
    raise ValueError("include must be one of: 'short', 'long', 'all', 'custom'")

# ----------------------------- Coloring helpers -----------------------------

def rand_color(alpha: float = 1.0) -> tuple[float, float, float, float]:
    """Random RGBA in [0,1]."""
    return (random.random(), random.random(), random.random(), alpha)

def color_for_pair(pair: Pair,
                   mode: str = "random",
                   custom: dict[Pair, str] | None = None) -> str:
    """
    Return a color for a given diagonal pair.
      - mode="random"  : random color
      - mode="type"    : 'short' vs 'long' fixed colors
      - mode="custom"  : use `custom` dict mapping (i,j) -> color
    """
    i, j = pair
    if mode == "random":
        r, g, b, _ = rand_color()
        # matplotlib accepts hex or RGB tuples; return tuple for convenience
        return (r, g, b)
    if mode == "type":
        step = (j - i) % 6
        return "#E4572E" if step in (2, 4) else "#3B78FF"  # short:red, long:blue
    if mode == "custom":
        if not custom:
            return "#000000"
        # ensure canonical key
        key = (min(i, j), max(i, j))
        return custom.get(key, "#000000")
    raise ValueError("color mode must be 'random', 'type', or 'custom'")

# ----------------------------- Drawing functions ----------------------------

def draw_hex(ax,
             verts: list[Point],
             edgecolor: str = "black",
             facecolor: str | None = None,
             linewidth: float = 1.5):
    """Draw the hexagon outline (and optional fill)."""
    poly = Polygon(verts, closed=True,
                   edgecolor=edgecolor,
                   facecolor=(facecolor if facecolor else "none"),
                   linewidth=linewidth, joinstyle="round")
    ax.add_patch(poly)

def draw_wedges(ax,
                verts: list[Point],
                colors: Iterable | None = None,
                alpha: float = 0.9):
    """
    Fill the 6 triangular wedges (center-vi-v(i+1)).
    If `colors` is None, randomize each wedge color.
    """
    c = hex_center(verts)
    for i in range(6):
        tri = [c, verts[i], verts[(i + 1) % 6]]
        color = next(colors) if colors else rand_color(alpha)
        poly = Polygon(tri, closed=True, edgecolor="none", facecolor=color)
        ax.add_patch(poly)

def draw_diagonals(ax,
                   verts: list[Point],
                   pairs: Iterable[Pair],
                   linewidth: float = 8.0,
                   color_mode: str = "random",
                   custom_colors: dict[Pair, str] | None = None,
                   alpha: float = 1.0):
    """Draw thick colored diagonals according to `pairs`."""
    for pair in pairs:
        i, j = pair
        (x1, y1), (x2, y2) = verts[i], verts[j]
        col = color_for_pair(pair, mode=color_mode, custom=custom_colors)
        ax.plot([x1, x2], [y1, y2],
                linewidth=linewidth,
                solid_capstyle="round",
                color=(*col, alpha) if isinstance(col, tuple) else col)

# ----------------------------- Grid helpers ---------------------------------

def hex_grid_centers(n_rows: int, n_cols: int, r: float,
                     start: Point = (0.0, 0.0),
                     flat_top: bool = True) -> list[Point]:
    """
    Return centers for a hex grid (odd-r offset).
    Spacing formulas differ by orientation.
    """
    cx0, cy0 = start
    centers = []
    if flat_top:
        dx = 1.5 * r
        dy = math.sqrt(3) * r
        for row in range(n_rows):
            for col in range(n_cols):
                cx = cx0 + col * dx
                cy = cy0 + row * dy + (dy / 2.0 if (col % 2) else 0.0)
                centers.append((cx, cy))
    else:
        dx = math.sqrt(3) * r
        dy = 1.5 * r
        for row in range(n_rows):
            for col in range(n_cols):
                cx = cx0 + col * dx + (dx / 2.0 if (row % 2) else 0.0)
                cy = cy0 + row * dy
                centers.append((cx, cy))
    return centers

# ----------------------------- Main API -------------------------------------

def create_hex_image(filename: str = "hex.png",
                     n_rows: int = 1,
                     n_cols: int = 1,
                     r: float = 1.0,
                     flat_top: bool = True,
                     include: str = "all",          # 'short' | 'long' | 'all' | 'custom'
                     pairs: Iterable[Pair] | None = None,
                     draw_hex_outline: bool = True,
                     hex_edgecolor: str = "black",
                     hex_facecolor: str | None = None,
                     hex_linewidth: float = 1.5,
                     draw_wedge_fill: bool = False,
                     diagonal_linewidth: float = 8.0,
                     diagonal_color_mode: str = "random",  # 'random' | 'type' | 'custom'
                     diagonal_custom_colors: dict[Pair, str] | None = None,
                     background: str = "white",
                     dpi: int = 300,
                     seed: int | None = None):
    """
    Build and save an image with a hex grid and configurable diagonals/colors.
    """
    if seed is not None:
        random.seed(seed)

    # Estimate figure size so hexes appear nicely spaced
    pad = 0.2 * r
    centers = hex_grid_centers(n_rows, n_cols, r, start=(0.0, 0.0), flat_top=flat_top)
    xs = [c[0] for c in centers]
    ys = [c[1] for c in centers]
    min_x, max_x = min(xs) - r - pad, max(xs) + r + pad
    min_y, max_y = min(ys) - r - pad, max(ys) + r + pad
    width_in = max(2.0, (max_x - min_x) * 1.1)
    height_in = max(2.0, (max_y - min_y) * 1.1)

    fig, ax = plt.subplots(figsize=(width_in, height_in), dpi=dpi)
    fig.patch.set_facecolor(background)
    ax.set_facecolor(background)

    chosen_pairs = diagonal_pairs(include, pairs)

    for center in centers:
        verts = regular_hex_vertices(center, r, flat_top=flat_top)

        if draw_wedge_fill:
            draw_wedges(ax, verts, colors=None, alpha=0.95)

        if draw_hex_outline:
            draw_hex(ax, verts, edgecolor=hex_edgecolor, facecolor=hex_facecolor,
                     linewidth=hex_linewidth)

        draw_diagonals(ax, verts, chosen_pairs,
                       linewidth=diagonal_linewidth,
                       color_mode=diagonal_color_mode,
                       custom_colors=diagonal_custom_colors,
                       alpha=1.0)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(filename, dpi=dpi, facecolor=background, bbox_inches="tight")
    plt.close(fig)

# ----------------------------- Examples -------------------------------------

if __name__ == "__main__":
    # Example 1: single hex, only LONG diagonals, random colors
    create_hex_image(
        filename="hex_long.png",
        n_rows=1, n_cols=1, r=1.5, flat_top=True,
        include="long",                   # only opposite-vertex diagonals (3)
        draw_wedge_fill=False,            # set True to color the 6 wedges
        diagonal_linewidth=10.0,
        diagonal_color_mode="random",     # 'random' | 'type' | 'custom'
        seed=42
    )

    # Example 2: 3x3 grid, ALL diagonals (9 per hex), random colors
    create_hex_image(
        filename="hex_grid_all.png",
        n_rows=3, n_cols=3, r=1.0, flat_top=True,
        include="all",
        draw_wedge_fill=False,
        diagonal_linewidth=6.0,
        diagonal_color_mode="random",
        seed=7
    )

    # Example 3: single hex, CUSTOM diagonal list + CUSTOM colors
    custom_pairs = [(0, 2), (1, 4), (0, 3)]  # two short, one long
    custom_colors = {
        (0, 2): "#D7263D",
        (1, 4): "#1B998B",
        (0, 3): "#2E294E"
    }
    create_hex_image(
        filename="hex_custom.png",
        n_rows=1, n_cols=1, r=1.8, flat_top=False,
        include="custom", pairs=custom_pairs,
        draw_wedge_fill=True,             # fills 6 wedges under the diagonals
        hex_facecolor=None,               # or e.g. "#f5f5f5"
        diagonal_linewidth=12.0,
        diagonal_color_mode="custom",
        diagonal_custom_colors=custom_colors,
        seed=123
    )


TypeError: __array_wrap__() argument 1 must be numpy.ndarray, not numpy.ndarray

Error in callback <function _draw_all_if_interactive at 0x000001BA220C29E0> (for post_execute):


ValueError: object __array__ method not producing an array

ImportError: cannot import name 'ERR_IGNORE' from 'numpy.core.umath' (c:\Users\SemKj\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\umath.py)

<Figure size 1188x1188 with 1 Axes>